In [1]:
from src.parameter_handler import ConfigManager, SimulationState
from omegaconf import OmegaConf
import numpy as np
from src.config import PathConfigs
from src.barrier_calibration import calibrate_wire_cutters
from src import helpers as hp
import src.barrier_calibration as bc
import xarray as xr

In [2]:
yamlpath = PathConfigs.PARAMETERS/"fixing_code.yaml"  

In [3]:
config = ConfigManager.initialize(yamlpath)
state = SimulationState(config)

geom = config.simulation.geometry
disorder = config.simulation.disorder
res = config.simulation.resolution
calib = config.simulation.barrier_calibration 

Vdis = state.Vdisx * 1.2

state.__dict__.keys()

mu = 2

Vz_var = state.params_list[:,2]
mu_var = state.params_list[:,1]

mu_set = list(set(mu_var))
Vz_set = list(set(Vz_var))

temps = [0,10,30,40]

state.target_GNs

Derived Hopping Parameter t: 16.5651


array([0.01, 0.05, 0.1 , 0.14, 0.19, 0.23, 0.25, 0.28, 0.32, 0.35, 0.41,
       0.45, 0.5 , 0.55, 0.58, 0.63, 0.65, 0.67, 0.72, 0.76, 0.81, 0.85,
       0.95])

In [9]:
Gmatarr = hp.initialize_conductance_dataarray(
    target_temps=temps,
    G_n_values = state.target_GNs,
    mu_set=mu_set,
    Vz_set=Vz_set

)

Gmatarr.sel(temperature = 30, swept_barrier = "left", G_n = [0.01], chemical_potential = 0.2, zeeman_field = 0.6 )

<xarray.DataArray 'barrier sweep conductance array' (G_n: 1, lead_row: 2,
                                                     lead_col: 2)>
array([[[0., 0.],
        [0., 0.]]])
Coordinates:
    temperature         int64 30
    swept_barrier       <U5 'left'
  * G_n                 (G_n) float64 0.01
  * lead_row            (lead_row) <U5 'left' 'right'
  * lead_col            (lead_col) <U5 'left' 'right'
    chemical_potential  float64 0.2
    zeeman_field        float64 0.6
Attributes:
    units:              e^2/h
    temperature_units:  mK

In [ ]:
target_gn = 1.5

ul = bc.calibrate_single_barrier(
    target_gn=target_gn,
    side = "left",
    calibration_cfg=calib,
    geometry_cfg=geom,
    Vdisx=Vdis,
    t = state.t,
    alpha=state.alpha,
    mu_leads = state.t,
    mu_points = [mu]
)

ur = bc.calibrate_single_barrier(
    target_gn=target_gn,
    side = "right",
    calibration_cfg=calib,
    geometry_cfg=geom,
    Vdisx=Vdis,
    t = state.t,
    alpha=state.alpha,
    mu_leads = state.t,
    mu_points = [mu]
)

print(f"UL: {ul}    UR: {ur}")

In [ ]:
syst = hp.build_system_exact(
    t=state.t,
    mu=mu,
    gamma=geom.gamma,
    Delta0=geom.Delta_0,
    V_z=0.0,
    alpha=state.alpha,
    Ls=geom.Ls,
    Vdisx=Vdis,
    barrier_l=ul,
    barrier_r=ur,
    Ln = geom.Ln,
    Lb = geom.Lb
)

G_mat = hp.calc_conductance_matrix_exact(syst, E=0.5)

np.isclose(G_mat[0,0], G_mat[1,1], atol = 0.1) 


G_mat

In [ ]:
syst = hp.build_system_exact(
    t=state.t,
    mu=mu,
    gamma=geom.gamma,
    Delta0=geom.Delta_0,
    V_z=0.0,
    alpha=state.alpha,
    Ls=geom.Ls,
    Vdisx=Vdis,
    barrier_l=ul,
    barrier_r=ur,
    Ln = geom.Ln,
    Lb = geom.Lb
)

G_mat = hp.calc_conductance_matrix_exact(syst, E=0.0)

G_mat

In [ ]:
syst = hp.build_system(
    t=state.t,
    mu=mu,
    gamma=geom.gamma,
    Delta0=geom.Delta_0,
    V_z=0.0,
    alpha=state.alpha,
    Ls=geom.Ls,
    Vdisx=Vdis,
    barrier_l=ul,
    barrier_r=ur,
    Ln = geom.Ln,
    Lb = geom.Lb,
    mu_leads=state.t

)

G_mat_renorm = hp.calc_conductance_matrix(syst, eng=0.0000000000)

all(np.isclose(G_mat_renorm, G_mat).flatten() )

In [ ]:
vdis_for_calib = state.Vdisx * float(disorder.V0) 
calibrated_cutters = bc.calibrate_wire_cutters(
    targets = np.asarray([0.2, 0.3]),
    calibration_cfg=calib,
    geometry_cfg=geom,
    Vdisx=vdis_for_calib,
    t=state.t,
    alpha=state.alpha,
    mu_leads=state.mu_leads,
)

In [ ]:
calibrated_cutters